# Platform Statistics
Extract statistics for specific platforms.

## Setup and connect to database

In [ ]:
import sqlite3
import pandas as pd

DB_FILE = 'socat_kpi.sqlite'
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

## Platforms
Specify the platforms that we are interested in. This consist of the platform name and ICES platform codes (i.e. the first characters of the EXPO Code).

Note that the codes must be in a list - this allows us to account for stations that have changed code, or if we want to combine stations.


In [ ]:
stations = {
    'G O Sars': ['58GS', '58G2'],
    'Finnmaid': ['34FM'],
    'Tukuma Arctica': ['26NA', '26RA'],
    'Polarstern': ['06AQ'],
    'Simon Stevin': ['11SS'],
    'San Lorenzo Maersk': ['65DK', '26T3'],
    'Sea-Cargo Express': ['BHNQ', 'MLSC'],
    'Italian moorings': ['48MB'],
    'Porcupine Abyssal Plain': ['74FS'],
    'Thornton Buoy': ['1199'],
    'Atlantic Sail': ['74WX']
}

# Validate platform info
for (station, codes) in stations.items():
    if not isinstance(codes, list):
        raise ValueError(f'Entry for {station} must be a list')
    elif len(codes) == 0:
        raise ValueError(f'Code list for {station} is empty')


## Number of days of with observations for each platform in each year

In [ ]:
required_years = [2022, 2023, 2024, 2025]

observation_days = dict()

df = pd.DataFrame(columns=['Station'] + required_years)

for (station, codes) in stations.items():
    print(station)
    cur.execute(f'SELECT year, count(DISTINCT(month_day)) FROM socat WHERE platform_code IN ({','.join(f'\'{code}\'' for code in codes)}) AND year IN ({','.join(str(year) for year in required_years)}) GROUP BY year ORDER BY year ASC')

    counts = [0] * len(required_years)
    for row in cur.fetchall():
        year_index = required_years.index(row[0])
        counts[year_index] = row[1]
            
    df.loc[len(df)] = [station] + counts

df = df.sort_values(by=['Station'])
df
                

# Shut down and tidy up

In [ ]:
cur.close()
conn.close()